# BAYA v1.1 — EfficientNet-B0–BiLSTM Real-Handwriting Refinement

This notebook refines the already-selected **BAYA EfficientNet-B0–BiLSTM v1.0** model using:

- the original **4,410-sample** dataset,
- the new real BAYA handwriting dataset (**about 15 samples × 63 classes = 945 samples**),
- conservative **synchronized image + sequence augmentation** on the training split only,
- a protected validation/test split for the new handwriting,
- the exact v1.0 preprocessing contract,
- low-learning-rate fine-tuning,
- explicit **v1.0 vs v1.1** evaluation.

## Important methodology

This notebook **does not restart the original three-model comparative analysis**. It treats EfficientNet-B0–BiLSTM as the already-selected model and performs a **post-selection refinement experiment**.

The original v1.0 model is never overwritten.

### Required files

You will upload:

1. the original dataset ZIP (the same dataset used for v1.0 training),
2. the new real-handwriting dataset ZIP,
3. `Baybayin_EfficientNetB0_BiLSTM_1.0.keras`.

The notebook is intentionally strict. It will stop if the original v1.0 test result cannot be reproduced closely enough, because that would indicate a dataset/split/preprocessing mismatch.

In [ ]:
# ============================================================
# STEP 1 — Runtime / reproducibility
# ============================================================

import os
import sys
import json
import math
import random
import shutil
import hashlib
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras

from PIL import Image

try:
    import cv2
except ImportError:
    import subprocess
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "opencv-python-headless"
    ])
    import cv2

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
)

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("Python:", sys.version.split()[0])
print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("OpenCV:", cv2.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

if not tf.config.list_physical_devices("GPU"):
    print("\nWARNING: No GPU detected. Training will work, but it may be slow.")

In [ ]:
# ============================================================
# STEP 2 — Experiment configuration
# ============================================================

# ----- Exact selected-model input contract -----
IMG_HEIGHT = 96
IMG_WIDTH = 96
IMG_CHANNELS = 1

SEQ_FEATURES = 6
MAX_SEQ_LEN = 512

IMAGE_PADDING_RATIO = 0.15
INK_THRESHOLD = 0.08
TRAJECTORY_PADDING_RATIO = 0.15

EXPECTED_CLASSES = 63
EXPECTED_OLD_SAMPLES = 4410
EXPECTED_NEW_SAMPLES = 63 * 15   # 945, based on the current collection

# ----- Original v1.0 benchmark -----
EXPECTED_V1_0_OLD_TEST_ACCURACY = 618 / 630  # 0.980952...
V1_0_REPRO_TOLERANCE = 0.005

# ----- New-data splitting -----
# Writer-based splitting is preferred when the export contains >= 3
# usable writer IDs and the resulting splits retain all 63 classes.
# Otherwise, the notebook falls back to deterministic per-class
# 60/20/20 splitting.
NEW_TRAIN_FRACTION = 0.60
NEW_VAL_FRACTION = 0.20
NEW_TEST_FRACTION = 0.20

# ----- Fine-tuning -----
STAGE_A_EPOCHS = 5
STAGE_A_LR = 5e-5

STAGE_B_EPOCHS = 18
STAGE_B_LR = 1e-5
FINE_TUNE_LAST_N = 20

BATCH_SIZE = 32

# ----- Conservative synchronized augmentation -----
# Every augmented image and sequence receives THE SAME affine transform.
# Validation and test data are NEVER augmented.
AUGMENT_COPIES_PER_OLD_TRAIN_SAMPLE = 1
AUGMENT_COPIES_PER_NEW_TRAIN_SAMPLE = 1

AUG_ROTATION_DEG = 5.0
AUG_SCALE_MIN = 0.95
AUG_SCALE_MAX = 1.05
AUG_TRANSLATION_PX = 2.0

# Give new real handwriting slightly more influence during refinement,
# without discarding the original training distribution.
OLD_SAMPLE_WEIGHT = 1.0
NEW_SAMPLE_WEIGHT = 1.5

OLD_VAL_WEIGHT = 1.0
NEW_VAL_WEIGHT = 2.0

# ----- Paths -----
ROOT = Path("/content/baya_v1_1")
UPLOAD_DIR = ROOT / "uploads"
OLD_EXTRACT_DIR = ROOT / "old_extracted"
NEW_EXTRACT_DIR = ROOT / "new_extracted"
OUTPUT_DIR = ROOT / "output"

for p in [UPLOAD_DIR, OLD_EXTRACT_DIR, NEW_EXTRACT_DIR, OUTPUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

V1_MODEL_PATH = UPLOAD_DIR / "Baybayin_EfficientNetB0_BiLSTM_1.0.keras"
OLD_ZIP_PATH = UPLOAD_DIR / "BAYA_DATASETS_original.zip"
NEW_ZIP_PATH = UPLOAD_DIR / "BAYA_DATASETS_new_real.zip"

STAGE_A_PATH = OUTPUT_DIR / "Baybayin_EfficientNetB0_BiLSTM_1.1_stageA.keras"
STAGE_B_PATH = OUTPUT_DIR / "Baybayin_EfficientNetB0_BiLSTM_1.1_stageB.keras"
FINAL_V1_1_PATH = OUTPUT_DIR / "Baybayin_EfficientNetB0_BiLSTM_1.1.keras"

print("Working directory:", ROOT)

## Upload the three required files

The file picker is run three times so there is no ambiguity about which file is which.

For the new dataset, the ZIP may contain either:

- nested export ZIPs,
- one normal export folder,
- or multiple folders containing `attempts.json`.

The loader searches recursively and standardizes the exports.

In [ ]:
# ============================================================
# STEP 3 — Upload model + original dataset + new dataset
# ============================================================

from google.colab import files

def upload_exactly_one(target_path: Path, description: str):
    print("\n" + "=" * 72)
    print(description)
    print("=" * 72)

    uploaded = files.upload()

    if len(uploaded) != 1:
        raise ValueError(
            f"Please upload exactly ONE file for: {description}. "
            f"Received {len(uploaded)} files."
        )

    source_name, data = next(iter(uploaded.items()))
    target_path.write_bytes(data)

    print("Uploaded:", source_name)
    print("Saved as:", target_path)
    print("Size:", f"{target_path.stat().st_size / (1024*1024):.2f} MB")

    return target_path

if not V1_MODEL_PATH.exists():
    upload_exactly_one(
        V1_MODEL_PATH,
        "UPLOAD 1/3 — Original v1.0 Keras model"
    )

if not OLD_ZIP_PATH.exists():
    upload_exactly_one(
        OLD_ZIP_PATH,
        "UPLOAD 2/3 — Original 4,410-sample BAYA dataset ZIP"
    )

if not NEW_ZIP_PATH.exists():
    upload_exactly_one(
        NEW_ZIP_PATH,
        "UPLOAD 3/3 — New real-handwriting dataset ZIP (~945 samples)"
    )

print("\nAll required files are present.")

In [ ]:
# ============================================================
# STEP 4 — ZIP extraction helpers
# ============================================================

def safe_extract_zip(zip_path: Path, destination: Path):
    destination.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as zf:
        base = destination.resolve()

        for member in zf.infolist():
            member_path = (destination / member.filename).resolve()

            if not str(member_path).startswith(str(base)):
                raise ValueError(
                    f"Unsafe ZIP member path detected: {member.filename}"
                )

        zf.extractall(destination)


def prepare_dataset_zip(zip_path: Path, destination: Path, tag: str):
    if destination.exists():
        shutil.rmtree(destination)

    destination.mkdir(parents=True, exist_ok=True)

    outer_dir = destination / "outer"
    exports_dir = destination / "exports"

    safe_extract_zip(zip_path, outer_dir)

    inner_zips = sorted(outer_dir.rglob("*.zip"))

    if inner_zips:
        exports_dir.mkdir(parents=True, exist_ok=True)

        for i, inner_zip in enumerate(inner_zips):
            safe_stem = (
                inner_zip.stem
                .replace(" ", "_")
                .replace("/", "_")
                .replace("\\", "_")
            )

            export_dir = exports_dir / f"{i:03d}_{safe_stem}"
            safe_extract_zip(inner_zip, export_dir)

        scan_root = exports_dir
        mode = "nested-export-zips"
    else:
        scan_root = outer_dir
        mode = "direct-extraction"

    attempts_files = sorted(scan_root.rglob("attempts.json"))

    if not attempts_files:
        raise FileNotFoundError(
            f"No attempts.json found after extracting {zip_path.name}"
        )

    print(
        f"{tag}: extraction mode={mode}, "
        f"attempts.json files={len(attempts_files)}"
    )

    return scan_root


old_scan_root = prepare_dataset_zip(
    OLD_ZIP_PATH,
    OLD_EXTRACT_DIR,
    "OLD"
)

new_scan_root = prepare_dataset_zip(
    NEW_ZIP_PATH,
    NEW_EXTRACT_DIR,
    "NEW"
)

In [ ]:
# ============================================================
# STEP 5 — Read attempts.json into a standardized dataframe
# ============================================================

WRITER_KEYS = [
    "writerId",
    "writer_id",
    "participantId",
    "participant_id",
    "studentId",
    "student_id",
    "learnerId",
    "learner_id",
    "userId",
    "user_id",
]

def first_nonempty(rec, keys):
    for key in keys:
        value = rec.get(key)

        if value is not None and str(value).strip() not in {"", "None", "null"}:
            return str(value)

    return None


def infer_source_export(attempts_file: Path, scan_root: Path):
    try:
        relative = attempts_file.parent.relative_to(scan_root)
        text = str(relative).replace("\\", "/")
    except Exception:
        text = attempts_file.parent.name

    return text if text not in {"", "."} else attempts_file.parent.name


def load_attempt_records(scan_root: Path, dataset_tag: str):
    records = []

    attempts_files = sorted(scan_root.rglob("attempts.json"))

    for attempts_file in attempts_files:
        export_root = attempts_file.parent
        source_export = infer_source_export(attempts_file, scan_root)

        with open(attempts_file, "r", encoding="utf-8") as f:
            attempts = json.load(f)

        if not isinstance(attempts, list):
            raise ValueError(
                f"{attempts_file} must contain a JSON array of attempts."
            )

        for rec in attempts:
            attempt_id = rec.get("attemptId")
            label = rec.get("targetClass")

            image_uri = rec.get("imageUri")

            if image_uri:
                image_path = export_root / image_uri
            else:
                image_path = export_root / "images" / f"attempt_{attempt_id}.png"

            writer_id = first_nonempty(rec, WRITER_KEYS)

            records.append({
                "dataset_tag": dataset_tag,
                "source_export": f"{dataset_tag}:{source_export}",
                "attempt_id": attempt_id,
                "label": label,
                "image_path": str(image_path),
                "canvas_width": rec.get("canvasWidth"),
                "canvas_height": rec.get("canvasHeight"),
                "duration_ms": rec.get("durationMs"),
                "strokes": rec.get("strokes", []),
                "writer_id": writer_id,
            })

    df = pd.DataFrame(records)

    if df.empty:
        raise ValueError(f"No attempts loaded for dataset tag={dataset_tag}")

    return df


old_df = load_attempt_records(old_scan_root, "old")
new_df = load_attempt_records(new_scan_root, "new")

print("Old samples:", len(old_df))
print("New samples:", len(new_df))
print("Old classes:", old_df["label"].nunique())
print("New classes:", new_df["label"].nunique())
print("New explicit writer IDs:", new_df["writer_id"].dropna().nunique())

In [ ]:
# ============================================================
# STEP 6 — Dataset validation
# ============================================================

def point_count(strokes):
    return sum(
        len(stroke.get("points", []))
        for stroke in (strokes or [])
    )

def stroke_count(strokes):
    return sum(
        1
        for stroke in (strokes or [])
        if len(stroke.get("points", [])) > 0
    )


def validate_dataset(df, name, expected_samples=None):
    print("\n" + "=" * 72)
    print(f"VALIDATING {name}")
    print("=" * 72)

    missing_labels = int(df["label"].isna().sum())

    df["image_exists"] = df["image_path"].apply(
        lambda p: Path(p).exists()
    )

    missing_images = int((~df["image_exists"]).sum())

    duplicate_attempts = int(
        df.duplicated(
            ["source_export", "attempt_id"]
        ).sum()
    )

    df["point_count"] = df["strokes"].apply(point_count)
    df["stroke_count"] = df["strokes"].apply(stroke_count)

    print("Samples:", len(df))
    print("Classes:", df["label"].nunique())
    print("Exports/sessions:", df["source_export"].nunique())
    print("Explicit writers:", df["writer_id"].dropna().nunique())
    print("Missing labels:", missing_labels)
    print("Missing images:", missing_images)
    print("Duplicate attempts:", duplicate_attempts)

    if expected_samples is not None and len(df) != expected_samples:
        print(
            f"WARNING: Expected about {expected_samples} samples, "
            f"but found {len(df)}."
        )

    if missing_labels:
        raise ValueError(f"{name}: missing targetClass labels.")

    if missing_images:
        preview = (
            df.loc[~df["image_exists"], "image_path"]
            .head(10)
            .tolist()
        )
        raise FileNotFoundError(
            f"{name}: {missing_images} image files are missing. "
            f"Examples: {preview}"
        )

    if duplicate_attempts:
        raise ValueError(
            f"{name}: duplicate (source_export, attempt_id) pairs found."
        )

    if df["label"].nunique() != EXPECTED_CLASSES:
        raise ValueError(
            f"{name}: expected {EXPECTED_CLASSES} classes, "
            f"found {df['label'].nunique()}."
        )

    class_counts = (
        df["label"]
        .astype(str)
        .value_counts()
        .sort_index()
    )

    print("\nSamples/class summary:")
    print(class_counts.describe())

    return class_counts


old_counts = validate_dataset(
    old_df,
    "ORIGINAL DATASET",
    expected_samples=EXPECTED_OLD_SAMPLES
)

new_counts = validate_dataset(
    new_df,
    "NEW REAL-HANDWRITING DATASET",
    expected_samples=EXPECTED_NEW_SAMPLES
)

if not set(old_df["label"].astype(str)) == set(new_df["label"].astype(str)):
    missing_new = sorted(
        set(old_df["label"].astype(str)) -
        set(new_df["label"].astype(str))
    )
    extra_new = sorted(
        set(new_df["label"].astype(str)) -
        set(old_df["label"].astype(str))
    )

    raise ValueError(
        "Old/new label sets differ. "
        f"Missing in new={missing_new}, extra in new={extra_new}"
    )

if new_counts.nunique() != 1:
    print("\nNOTE: New dataset is not perfectly class-balanced.")
else:
    print(
        f"\nNew dataset is balanced: "
        f"{int(new_counts.iloc[0])} samples/class."
    )

In [ ]:
# ============================================================
# STEP 7 — Authoritative label mapping
# ============================================================

classes = sorted(
    old_df["label"]
    .astype(str)
    .unique()
)

label_to_id = {
    label: idx
    for idx, label in enumerate(classes)
}

id_to_label = {
    idx: label
    for label, idx in label_to_id.items()
}

NUM_CLASSES = len(classes)

if NUM_CLASSES != 63:
    raise ValueError(f"Expected 63 classes, found {NUM_CLASSES}")

for frame in [old_df, new_df]:
    frame["label"] = frame["label"].astype(str)
    frame["label_id"] = (
        frame["label"]
        .map(label_to_id)
        .astype("int32")
    )

print("Number of classes:", NUM_CLASSES)
print("\nFirst 20 mappings:")

for label in classes[:20]:
    print(f"{label:12s} -> {label_to_id[label]}")

In [ ]:
# ============================================================
# STEP 8 — EXACT v1.0 image preprocessing
# ============================================================

def load_image_array(image_path):
    # 1. Load and place transparent images on white.
    img = Image.open(image_path).convert("RGBA")
    bg = Image.new("RGBA", img.size, (255, 255, 255, 255))
    bg.alpha_composite(img)

    # 2. Convert to grayscale.
    gray = bg.convert("L")
    arr = np.asarray(gray, dtype=np.float32) / 255.0

    # 3. white background -> 0; black handwriting -> 1
    ink = 1.0 - arr

    # 4. Detect all visible handwriting.
    mask = ink > INK_THRESHOLD

    if np.any(mask):
        ys, xs = np.where(mask)

        y0, y1 = int(ys.min()), int(ys.max()) + 1
        x0, x1 = int(xs.min()), int(xs.max()) + 1

        crop = ink[y0:y1, x0:x1]

        crop_h, crop_w = crop.shape
        max_dim = max(crop_h, crop_w)

        padding = max(
            2,
            int(round(max_dim * IMAGE_PADDING_RATIO))
        )

        square_side = max_dim + 2 * padding

        square = np.zeros(
            (square_side, square_side),
            dtype=np.float32
        )

        top = (square_side - crop_h) // 2
        left = (square_side - crop_w) // 2

        square[
            top:top + crop_h,
            left:left + crop_w
        ] = crop

    else:
        square = ink

    square_u8 = (
        np.clip(square * 255.0, 0, 255)
        .astype(np.uint8)
    )

    standardized = Image.fromarray(
        square_u8,
        mode="L"
    ).resize(
        (IMG_WIDTH, IMG_HEIGHT),
        Image.Resampling.BILINEAR,
    )

    result = (
        np.asarray(
            standardized,
            dtype=np.float32
        ) / 255.0
    )

    result = np.expand_dims(
        result,
        axis=-1
    )

    return result.astype(np.float32)

In [ ]:
# ============================================================
# STEP 9 — EXACT v1.0 sequence preprocessing
# ============================================================

def allocate_points_per_stroke(lengths, max_total):
    n = len(lengths)

    if n == 0:
        return []

    total = sum(lengths)

    if total <= max_total:
        return lengths.copy()

    min_each = 2 if max_total >= 2 * n else 1

    base = np.array(
        [min(min_each, L) for L in lengths],
        dtype=int
    )

    remaining = max_total - int(base.sum())

    if remaining <= 0:
        return base.tolist()

    capacity = np.array(
        [
            max(0, L - b)
            for L, b in zip(lengths, base)
        ],
        dtype=float,
    )

    if capacity.sum() == 0:
        return base.tolist()

    raw_extra = (
        remaining *
        capacity /
        capacity.sum()
    )

    extra = np.floor(
        raw_extra
    ).astype(int)

    alloc = base + extra

    leftover = (
        max_total -
        int(alloc.sum())
    )

    if leftover > 0:
        order = np.argsort(
            -(raw_extra - extra)
        )

        for idx in order:
            if leftover <= 0:
                break

            if alloc[idx] < lengths[idx]:
                alloc[idx] += 1
                leftover -= 1

    return alloc.tolist()


def select_evenly(points, target_count):
    if target_count >= len(points):
        return points

    if target_count <= 1:
        return [points[0]]

    indices = np.linspace(
        0,
        len(points) - 1,
        target_count
    )

    indices = np.unique(
        np.round(indices).astype(int)
    )

    if len(indices) < target_count:
        used = set(indices.tolist())

        unused = [
            i
            for i in range(len(points))
            if i not in used
        ]

        need = (
            target_count -
            len(indices)
        )

        indices = np.sort(
            np.concatenate([
                indices,
                np.asarray(
                    unused[:need],
                    dtype=int
                )
            ])
        )

    return [
        points[int(i)]
        for i in indices[:target_count]
    ]


def get_trajectory_transform(clean_strokes):
    all_points = [
        point
        for stroke in clean_strokes
        for point in stroke.get("points", [])
    ]

    if not all_points:
        return 0.0, 0.0, 1.0

    xs = np.asarray(
        [float(p["x"]) for p in all_points],
        dtype=np.float32
    )

    ys = np.asarray(
        [float(p["y"]) for p in all_points],
        dtype=np.float32
    )

    min_x, max_x = float(xs.min()), float(xs.max())
    min_y, max_y = float(ys.min()), float(ys.max())

    center_x = (min_x + max_x) / 2.0
    center_y = (min_y + max_y) / 2.0

    width = max_x - min_x
    height = max_y - min_y

    handwriting_size = max(
        width,
        height,
        1e-6
    )

    scale = (
        handwriting_size *
        (1.0 + 2.0 * TRAJECTORY_PADDING_RATIO)
    )

    return center_x, center_y, scale


def build_sequence(row, max_len=MAX_SEQ_LEN):
    strokes = row["strokes"]

    duration = max(
        float(row["duration_ms"] or 0.0),
        1.0
    )

    clean_strokes = [
        stroke
        for stroke in strokes
        if len(stroke.get("points", [])) > 0
    ]

    if not clean_strokes:
        return np.zeros(
            (max_len, SEQ_FEATURES),
            dtype=np.float32
        )

    center_x, center_y, scale = (
        get_trajectory_transform(
            clean_strokes
        )
    )

    lengths = [
        len(stroke["points"])
        for stroke in clean_strokes
    ]

    allocated = (
        allocate_points_per_stroke(
            lengths,
            max_len
        )
    )

    sampled_strokes = []

    for stroke, n_keep in zip(
        clean_strokes,
        allocated
    ):
        sampled_strokes.append(
            select_evenly(
                stroke["points"],
                n_keep
            )
        )

    features = []
    previous_global_t = None

    for stroke_points in sampled_strokes:
        previous_x = None
        previous_y = None

        for j, point in enumerate(
            stroke_points
        ):
            raw_x = float(point["x"])
            raw_y = float(point["y"])
            t = float(
                point.get("t", 0.0)
            )

            x = (
                (raw_x - center_x) /
                scale
            ) + 0.5

            y = (
                (raw_y - center_y) /
                scale
            ) + 0.5

            stroke_start = (
                1.0 if j == 0 else 0.0
            )

            if j == 0:
                dx = 0.0
                dy = 0.0
            else:
                dx = x - previous_x
                dy = y - previous_y

            if previous_global_t is None:
                dt_norm = 0.0
            else:
                dt_norm = (
                    max(
                        0.0,
                        t - previous_global_t
                    ) /
                    duration
                )

            features.append([
                np.clip(x, 0.0, 1.0),
                np.clip(y, 0.0, 1.0),
                dx,
                dy,
                np.clip(
                    dt_norm,
                    0.0,
                    1.0
                ),
                stroke_start,
            ])

            previous_x = x
            previous_y = y
            previous_global_t = t

    seq = np.asarray(
        features,
        dtype=np.float32
    )

    if len(seq) < max_len:
        padding = np.zeros(
            (
                max_len - len(seq),
                SEQ_FEATURES
            ),
            dtype=np.float32
        )

        seq = np.vstack([
            seq,
            padding
        ])

    seq = seq[:max_len]

    return seq.astype(np.float32)

In [ ]:
# ============================================================
# STEP 10 — Original dataset split (same 10/2/2 export-group logic)
# ============================================================

old_groups = sorted(
    old_df["source_export"].unique()
)

if len(old_groups) != 14:
    raise ValueError(
        "The original v1.0 split expects exactly 14 source exports. "
        f"Found {len(old_groups)}."
    )

rng_old = np.random.default_rng(SEED)

shuffled_old_groups = np.array(
    old_groups,
    dtype=object
)

rng_old.shuffle(
    shuffled_old_groups
)

old_train_groups = (
    shuffled_old_groups[:10]
    .tolist()
)

old_val_groups = (
    shuffled_old_groups[10:12]
    .tolist()
)

old_test_groups = (
    shuffled_old_groups[12:14]
    .tolist()
)

old_train_df = old_df[
    old_df["source_export"].isin(
        old_train_groups
    )
].copy()

old_val_df = old_df[
    old_df["source_export"].isin(
        old_val_groups
    )
].copy()

old_test_df = old_df[
    old_df["source_export"].isin(
        old_test_groups
    )
].copy()

print("Old train:", len(old_train_df))
print("Old val:", len(old_val_df))
print("Old test:", len(old_test_df))

assert len(old_train_df) == 3150
assert len(old_val_df) == 630
assert len(old_test_df) == 630

for name, part in [
    ("old_train", old_train_df),
    ("old_val", old_val_df),
    ("old_test", old_test_df),
]:
    missing = sorted(
        set(classes) -
        set(part["label"])
    )

    if missing:
        raise ValueError(
            f"{name} missing classes: {missing}"
        )

In [ ]:
# ============================================================
# STEP 11 — New real-data split
#
# Preferred:
#   writer-group holdout
#
# Fallback:
#   deterministic per-class 60/20/20
# ============================================================

def all_classes_present(frame):
    return (
        set(frame["label"]) ==
        set(classes)
    )


def try_writer_group_split(df):
    explicit = df[
        df["writer_id"].notna()
    ].copy()

    if len(explicit) != len(df):
        return None

    writers = sorted(
        explicit["writer_id"].unique()
    )

    if len(writers) < 3:
        return None

    rng = np.random.default_rng(SEED)

    writers = np.array(
        writers,
        dtype=object
    )

    rng.shuffle(writers)

    # Search simple writer-count cut points that approximate 60/20/20
    candidates = []

    for n_train in range(1, len(writers) - 1):
        for n_val in range(1, len(writers) - n_train):
            n_test = (
                len(writers) -
                n_train -
                n_val
            )

            if n_test < 1:
                continue

            train_w = writers[:n_train]
            val_w = writers[
                n_train:n_train + n_val
            ]
            test_w = writers[
                n_train + n_val:
            ]

            train = explicit[
                explicit["writer_id"].isin(
                    train_w
                )
            ]

            val = explicit[
                explicit["writer_id"].isin(
                    val_w
                )
            ]

            test = explicit[
                explicit["writer_id"].isin(
                    test_w
                )
            ]

            if not (
                all_classes_present(train) and
                all_classes_present(val) and
                all_classes_present(test)
            ):
                continue

            fractions = np.array([
                len(train),
                len(val),
                len(test)
            ]) / len(explicit)

            target = np.array([
                NEW_TRAIN_FRACTION,
                NEW_VAL_FRACTION,
                NEW_TEST_FRACTION
            ])

            error = float(
                np.abs(
                    fractions - target
                ).sum()
            )

            candidates.append((
                error,
                train.copy(),
                val.copy(),
                test.copy(),
                train_w.tolist(),
                val_w.tolist(),
                test_w.tolist(),
            ))

    if not candidates:
        return None

    candidates.sort(
        key=lambda x: x[0]
    )

    return candidates[0]


def stratified_per_class_split(df):
    rng = np.random.default_rng(SEED)

    train_parts = []
    val_parts = []
    test_parts = []

    for label in classes:
        part = (
            df[df["label"] == label]
            .copy()
            .reset_index(drop=True)
        )

        indices = np.arange(len(part))
        rng.shuffle(indices)

        part = part.iloc[
            indices
        ].reset_index(drop=True)

        n = len(part)

        if n < 5:
            raise ValueError(
                f"Class {label!r} has only {n} new samples. "
                "Need at least 5 for train/val/test splitting."
            )

        n_test = max(
            1,
            int(round(
                n * NEW_TEST_FRACTION
            ))
        )

        n_val = max(
            1,
            int(round(
                n * NEW_VAL_FRACTION
            ))
        )

        if n_test + n_val >= n:
            n_test = 1
            n_val = 1

        n_train = (
            n -
            n_val -
            n_test
        )

        train_parts.append(
            part.iloc[:n_train]
        )

        val_parts.append(
            part.iloc[
                n_train:n_train+n_val
            ]
        )

        test_parts.append(
            part.iloc[
                n_train+n_val:
            ]
        )

    train = pd.concat(
        train_parts,
        ignore_index=True
    )

    val = pd.concat(
        val_parts,
        ignore_index=True
    )

    test = pd.concat(
        test_parts,
        ignore_index=True
    )

    # Shuffle within each completed split.
    train = train.sample(
        frac=1,
        random_state=SEED
    ).reset_index(drop=True)

    val = val.sample(
        frac=1,
        random_state=SEED + 1
    ).reset_index(drop=True)

    test = test.sample(
        frac=1,
        random_state=SEED + 2
    ).reset_index(drop=True)

    return train, val, test


writer_split = try_writer_group_split(
    new_df
)

if writer_split is not None:
    (
        _,
        new_train_df,
        new_val_df,
        new_test_df,
        new_train_writers,
        new_val_writers,
        new_test_writers,
    ) = writer_split

    NEW_SPLIT_STRATEGY = (
        "writer-group holdout"
    )

    print(
        "Using WRITER-GROUP split."
    )

    print(
        "Train writers:",
        new_train_writers
    )

    print(
        "Val writers:",
        new_val_writers
    )

    print(
        "Test writers:",
        new_test_writers
    )

else:
    (
        new_train_df,
        new_val_df,
        new_test_df
    ) = stratified_per_class_split(
        new_df
    )

    NEW_SPLIT_STRATEGY = (
        "per-class stratified fallback "
        "(NOT an unseen-writer evaluation)"
    )

    print(
        "WARNING: Writer-group split was not possible."
    )

    print(
        "Using deterministic per-class split instead."
    )

print("\nNew split strategy:", NEW_SPLIT_STRATEGY)
print("New train:", len(new_train_df))
print("New val:", len(new_val_df))
print("New test:", len(new_test_df))

for name, part in [
    ("new_train", new_train_df),
    ("new_val", new_val_df),
    ("new_test", new_test_df),
]:
    if not all_classes_present(part):
        raise ValueError(
            f"{name} does not contain all 63 classes."
        )

    counts = (
        part["label"]
        .value_counts()
    )

    print(
        f"{name}: per-class min={counts.min()}, "
        f"max={counts.max()}"
    )

In [ ]:
# ============================================================
# STEP 12 — Save split manifests BEFORE training
# ============================================================

split_dir = OUTPUT_DIR / "splits"
split_dir.mkdir(
    parents=True,
    exist_ok=True
)

def save_split_manifest(frame, name):
    columns = [
        "dataset_tag",
        "source_export",
        "writer_id",
        "attempt_id",
        "label",
        "label_id",
        "image_path",
        "duration_ms",
        "point_count",
        "stroke_count",
    ]

    existing = [
        c
        for c in columns
        if c in frame.columns
    ]

    path = (
        split_dir /
        f"{name}.csv"
    )

    frame[
        existing
    ].to_csv(
        path,
        index=False
    )

    return path


for frame, name in [
    (old_train_df, "old_train"),
    (old_val_df, "old_val"),
    (old_test_df, "old_test"),
    (new_train_df, "new_train"),
    (new_val_df, "new_val"),
    (new_test_df, "new_test"),
]:
    save_split_manifest(
        frame,
        name
    )

print("Saved split manifests to:", split_dir)

In [ ]:
# ============================================================
# STEP 13 — Convert dataframes to model arrays
# ============================================================

def dataframe_to_arrays(
    part_df,
    name="dataset"
):
    images = []
    sequences = []
    labels = []

    total = len(part_df)

    for pos, (_, row) in enumerate(
        part_df.iterrows(),
        start=1
    ):
        images.append(
            load_image_array(
                row["image_path"]
            )
        )

        sequences.append(
            build_sequence(
                row
            )
        )

        labels.append(
            int(row["label_id"])
        )

        if (
            pos % 250 == 0
            or pos == total
        ):
            print(
                f"{name}: processed "
                f"{pos}/{total}"
            )

    return (
        np.stack(
            images
        ).astype(np.float32),

        np.stack(
            sequences
        ).astype(np.float32),

        np.asarray(
            labels,
            dtype=np.int32
        ),
    )


Ximg_old_train, Xseq_old_train, y_old_train = (
    dataframe_to_arrays(
        old_train_df,
        "old_train"
    )
)

Ximg_old_val, Xseq_old_val, y_old_val = (
    dataframe_to_arrays(
        old_val_df,
        "old_val"
    )
)

Ximg_old_test, Xseq_old_test, y_old_test = (
    dataframe_to_arrays(
        old_test_df,
        "old_test"
    )
)

Ximg_new_train, Xseq_new_train, y_new_train = (
    dataframe_to_arrays(
        new_train_df,
        "new_train"
    )
)

Ximg_new_val, Xseq_new_val, y_new_val = (
    dataframe_to_arrays(
        new_val_df,
        "new_val"
    )
)

Ximg_new_test, Xseq_new_test, y_new_test = (
    dataframe_to_arrays(
        new_test_df,
        "new_test"
    )
)

print("\nShapes:")
print("old_train:", Ximg_old_train.shape, Xseq_old_train.shape)
print("old_val:  ", Ximg_old_val.shape, Xseq_old_val.shape)
print("old_test: ", Ximg_old_test.shape, Xseq_old_test.shape)
print("new_train:", Ximg_new_train.shape, Xseq_new_train.shape)
print("new_val:  ", Ximg_new_val.shape, Xseq_new_val.shape)
print("new_test: ", Ximg_new_test.shape, Xseq_new_test.shape)

In [ ]:
# ============================================================
# STEP 14 — Evaluation helpers
# ============================================================

def predict_probs(
    model,
    Ximg,
    Xseq,
    batch_size=BATCH_SIZE
):
    probs = model.predict(
        {
            "image_input": Ximg,
            "sequence_input": Xseq,
        },
        batch_size=batch_size,
        verbose=0,
    )

    probs = np.asarray(
        probs,
        dtype=np.float32
    )

    if probs.shape != (
        len(Ximg),
        NUM_CLASSES
    ):
        raise ValueError(
            f"Unexpected prediction shape: {probs.shape}"
        )

    if not np.isfinite(
        probs
    ).all():
        raise ValueError(
            "Prediction output contains NaN/Inf."
        )

    return probs


def metrics_from_probs(
    y_true,
    probs
):
    y_pred = np.argmax(
        probs,
        axis=1
    )

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    (
        macro_precision,
        macro_recall,
        macro_f1,
        _
    ) = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )

    return {
        "accuracy": float(accuracy),
        "macro_precision": float(macro_precision),
        "macro_recall": float(macro_recall),
        "macro_f1": float(macro_f1),
        "y_pred": y_pred,
    }


def evaluate_model(
    model,
    Ximg,
    Xseq,
    y,
    title
):
    probs = predict_probs(
        model,
        Ximg,
        Xseq
    )

    m = metrics_from_probs(
        y,
        probs
    )

    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)

    print(
        f"Accuracy:        "
        f"{m['accuracy']:.4f} "
        f"({m['accuracy']*100:.2f}%)"
    )

    print(
        f"Macro Precision: "
        f"{m['macro_precision']:.4f}"
    )

    print(
        f"Macro Recall:    "
        f"{m['macro_recall']:.4f}"
    )

    print(
        f"Macro F1:        "
        f"{m['macro_f1']:.4f}"
    )

    m["probs"] = probs

    return m

In [ ]:
# ============================================================
# STEP 15 — Load v1.0 and verify its model contract
# ============================================================

v1_model = tf.keras.models.load_model(
    V1_MODEL_PATH,
    compile=False
)

print("Loaded:", V1_MODEL_PATH.name)

print("\nInputs:")

for tensor in v1_model.inputs:
    print(
        tensor.name,
        tensor.shape,
        tensor.dtype
    )

print("\nOutput:")

for tensor in v1_model.outputs:
    print(
        tensor.name,
        tensor.shape,
        tensor.dtype
    )

input_shapes = {
    tuple(
        int(d) if d is not None else None
        for d in tensor.shape
    )
    for tensor in v1_model.inputs
}

expected_input_shapes = {
    (None, 96, 96, 1),
    (None, 512, 6),
}

if input_shapes != expected_input_shapes:
    raise ValueError(
        f"Unexpected model inputs: {input_shapes}"
    )

output_shape = tuple(
    int(d) if d is not None else None
    for d in v1_model.outputs[0].shape
)

if output_shape != (
    None,
    NUM_CLASSES
):
    raise ValueError(
        f"Unexpected output shape: {output_shape}"
    )

print("\nModel contract is correct.")

In [ ]:
# ============================================================
# STEP 16 — v1.0 baseline BEFORE any refinement
# ============================================================

baseline_old_test = evaluate_model(
    v1_model,
    Ximg_old_test,
    Xseq_old_test,
    y_old_test,
    "v1.0 — ORIGINAL HELD-OUT TEST SET"
)

baseline_new_val = evaluate_model(
    v1_model,
    Ximg_new_val,
    Xseq_new_val,
    y_new_val,
    "v1.0 — NEW REAL-HANDWRITING VALIDATION"
)

baseline_new_test = evaluate_model(
    v1_model,
    Ximg_new_test,
    Xseq_new_test,
    y_new_test,
    "v1.0 — NEW REAL-HANDWRITING TEST"
)

baseline_gap = abs(
    baseline_old_test["accuracy"] -
    EXPECTED_V1_0_OLD_TEST_ACCURACY
)

print(
    "\nExpected original accuracy:",
    f"{EXPECTED_V1_0_OLD_TEST_ACCURACY:.6f}"
)

print(
    "Observed original accuracy:",
    f"{baseline_old_test['accuracy']:.6f}"
)

print(
    "Absolute difference:",
    f"{baseline_gap:.6f}"
)

if baseline_gap > V1_0_REPRO_TOLERANCE:
    raise RuntimeError(
        "STOP: v1.0 old-test accuracy was not reproduced closely enough. "
        "Do not fine-tune until the original dataset/split/preprocessing "
        "is verified."
    )

print("\nPASS: v1.0 baseline reproduction gate.")

## Synchronized augmentation

The augmentation below is intentionally conservative.

It applies **the same affine transform** to:

- the preprocessed `96×96×1` image, and
- the valid `(x_shape, y_shape)` sequence coordinates.

Then `dx/dy` are recomputed from the transformed coordinates.

It does **not** move a kudlit independently, flip the image, delete strokes, or change class semantics.

In [ ]:
# ============================================================
# STEP 17 — Synchronized image + sequence augmentation
# ============================================================

def make_affine_matrix(
    rng,
    size=96
):
    angle_deg = float(
        rng.uniform(
            -AUG_ROTATION_DEG,
            AUG_ROTATION_DEG
        )
    )

    sx = float(
        rng.uniform(
            AUG_SCALE_MIN,
            AUG_SCALE_MAX
        )
    )

    sy = float(
        rng.uniform(
            AUG_SCALE_MIN,
            AUG_SCALE_MAX
        )
    )

    tx = float(
        rng.uniform(
            -AUG_TRANSLATION_PX,
            AUG_TRANSLATION_PX
        )
    )

    ty = float(
        rng.uniform(
            -AUG_TRANSLATION_PX,
            AUG_TRANSLATION_PX
        )
    )

    theta = np.deg2rad(
        angle_deg
    )

    c = np.cos(theta)
    s = np.sin(theta)

    # A = rotation @ anisotropic scale
    A = np.array([
        [c * sx, -s * sy],
        [s * sx,  c * sy],
    ], dtype=np.float32)

    center = np.array([
        (size - 1) / 2.0,
        (size - 1) / 2.0,
    ], dtype=np.float32)

    translation = np.array([
        tx,
        ty
    ], dtype=np.float32)

    b = (
        center +
        translation -
        A @ center
    )

    M = np.concatenate(
        [
            A,
            b[:, None]
        ],
        axis=1
    ).astype(np.float32)

    params = {
        "angle_deg": angle_deg,
        "sx": sx,
        "sy": sy,
        "tx_px": tx,
        "ty_px": ty,
    }

    return M, params


def augment_pair(
    image,
    sequence,
    rng
):
    if image.shape != (
        IMG_HEIGHT,
        IMG_WIDTH,
        1
    ):
        raise ValueError(
            f"Bad image shape: {image.shape}"
        )

    if sequence.shape != (
        MAX_SEQ_LEN,
        SEQ_FEATURES
    ):
        raise ValueError(
            f"Bad sequence shape: {sequence.shape}"
        )

    M, params = make_affine_matrix(
        rng,
        size=IMG_WIDTH
    )

    # ---------- Image ----------
    warped = cv2.warpAffine(
        image[..., 0],
        M,
        (
            IMG_WIDTH,
            IMG_HEIGHT
        ),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0.0,
    )

    warped = np.clip(
        warped,
        0.0,
        1.0
    ).astype(np.float32)

    warped = warped[..., None]

    # ---------- Sequence ----------
    seq = sequence.copy().astype(
        np.float32
    )

    valid_mask = np.any(
        seq != 0.0,
        axis=1
    )

    valid_indices = np.where(
        valid_mask
    )[0]

    if len(valid_indices) > 0:
        raw_xy = (
            seq[
                valid_indices,
                0:2
            ].astype(np.float32) *
            float(IMG_WIDTH - 1)
        )

        ones = np.ones(
            (
                len(raw_xy),
                1
            ),
            dtype=np.float32
        )

        homo = np.concatenate(
            [
                raw_xy,
                ones
            ],
            axis=1
        )

        transformed_px = (
            homo @ M.T
        )

        transformed_xy = (
            transformed_px /
            float(IMG_WIDTH - 1)
        )

        # Preserve unclipped coordinates for dx/dy computation,
        # matching the training convention as closely as possible.
        seq[
            valid_indices,
            0
        ] = np.clip(
            transformed_xy[:, 0],
            0.0,
            1.0
        )

        seq[
            valid_indices,
            1
        ] = np.clip(
            transformed_xy[:, 1],
            0.0,
            1.0
        )

        previous_x = None
        previous_y = None

        for local_pos, row_idx in enumerate(
            valid_indices
        ):
            x_raw = float(
                transformed_xy[
                    local_pos,
                    0
                ]
            )

            y_raw = float(
                transformed_xy[
                    local_pos,
                    1
                ]
            )

            stroke_start = (
                seq[
                    row_idx,
                    5
                ] >= 0.5
            )

            if (
                stroke_start
                or previous_x is None
            ):
                seq[
                    row_idx,
                    2
                ] = 0.0

                seq[
                    row_idx,
                    3
                ] = 0.0
            else:
                seq[
                    row_idx,
                    2
                ] = (
                    x_raw -
                    previous_x
                )

                seq[
                    row_idx,
                    3
                ] = (
                    y_raw -
                    previous_y
                )

            previous_x = x_raw
            previous_y = y_raw

    # Padding must remain EXACT zero rows.
    seq[
        ~valid_mask
    ] = 0.0

    if not np.isfinite(
        warped
    ).all():
        raise ValueError(
            "Augmented image contains NaN/Inf."
        )

    if not np.isfinite(
        seq
    ).all():
        raise ValueError(
            "Augmented sequence contains NaN/Inf."
        )

    return warped, seq, params

In [ ]:
# ============================================================
# STEP 18 — Visual augmentation sanity check
# ============================================================

rng_preview = np.random.default_rng(
    SEED
)

sample_idx = 0

aug_img, aug_seq, aug_params = (
    augment_pair(
        Ximg_new_train[
            sample_idx
        ],
        Xseq_new_train[
            sample_idx
        ],
        rng_preview
    )
)

print("Augmentation parameters:")
print(json.dumps(
    aug_params,
    indent=2
))

print(
    "Original image range:",
    float(
        Ximg_new_train[
            sample_idx
        ].min()
    ),
    "to",
    float(
        Ximg_new_train[
            sample_idx
        ].max()
    )
)

print(
    "Augmented image range:",
    float(aug_img.min()),
    "to",
    float(aug_img.max())
)

plt.figure(
    figsize=(4, 4)
)

plt.imshow(
    aug_img[..., 0],
    cmap="gray"
)

plt.title(
    "One synchronized training augmentation"
)

plt.axis("off")
plt.show()

print(
    "Inspect this preview before continuing. "
    "Kudlit/strokes should still represent the SAME class."
)

In [ ]:
# ============================================================
# STEP 19 — Build augmented training arrays
# ============================================================

def make_augmented_copies(
    Ximg,
    Xseq,
    y,
    copies,
    seed
):
    if copies <= 0:
        return (
            np.empty(
                (
                    0,
                    IMG_HEIGHT,
                    IMG_WIDTH,
                    IMG_CHANNELS
                ),
                dtype=np.float32
            ),
            np.empty(
                (
                    0,
                    MAX_SEQ_LEN,
                    SEQ_FEATURES
                ),
                dtype=np.float32
            ),
            np.empty(
                (0,),
                dtype=np.int32
            ),
        )

    rng = np.random.default_rng(
        seed
    )

    out_img = []
    out_seq = []
    out_y = []

    total = len(y) * copies
    done = 0

    for copy_idx in range(copies):
        for i in range(len(y)):
            img_aug, seq_aug, _ = (
                augment_pair(
                    Ximg[i],
                    Xseq[i],
                    rng
                )
            )

            out_img.append(
                img_aug
            )

            out_seq.append(
                seq_aug
            )

            out_y.append(
                int(y[i])
            )

            done += 1

            if (
                done % 500 == 0
                or done == total
            ):
                print(
                    f"augmentation: "
                    f"{done}/{total}"
                )

    return (
        np.stack(
            out_img
        ).astype(np.float32),

        np.stack(
            out_seq
        ).astype(np.float32),

        np.asarray(
            out_y,
            dtype=np.int32
        ),
    )


print("Generating OLD training augmentations...")

(
    Ximg_old_aug,
    Xseq_old_aug,
    y_old_aug
) = make_augmented_copies(
    Ximg_old_train,
    Xseq_old_train,
    y_old_train,
    AUGMENT_COPIES_PER_OLD_TRAIN_SAMPLE,
    SEED + 100
)

print("\nGenerating NEW training augmentations...")

(
    Ximg_new_aug,
    Xseq_new_aug,
    y_new_aug
) = make_augmented_copies(
    Ximg_new_train,
    Xseq_new_train,
    y_new_train,
    AUGMENT_COPIES_PER_NEW_TRAIN_SAMPLE,
    SEED + 200
)

In [ ]:
# ============================================================
# STEP 20 — Assemble training / validation tensors + weights
# ============================================================

train_img_parts = [
    Ximg_old_train,
    Ximg_new_train,
]

train_seq_parts = [
    Xseq_old_train,
    Xseq_new_train,
]

train_y_parts = [
    y_old_train,
    y_new_train,
]

train_weight_parts = [
    np.full(
        len(y_old_train),
        OLD_SAMPLE_WEIGHT,
        dtype=np.float32
    ),

    np.full(
        len(y_new_train),
        NEW_SAMPLE_WEIGHT,
        dtype=np.float32
    ),
]

if len(y_old_aug) > 0:
    train_img_parts.append(
        Ximg_old_aug
    )
    train_seq_parts.append(
        Xseq_old_aug
    )
    train_y_parts.append(
        y_old_aug
    )
    train_weight_parts.append(
        np.full(
            len(y_old_aug),
            OLD_SAMPLE_WEIGHT,
            dtype=np.float32
        )
    )

if len(y_new_aug) > 0:
    train_img_parts.append(
        Ximg_new_aug
    )
    train_seq_parts.append(
        Xseq_new_aug
    )
    train_y_parts.append(
        y_new_aug
    )
    train_weight_parts.append(
        np.full(
            len(y_new_aug),
            NEW_SAMPLE_WEIGHT,
            dtype=np.float32
        )
    )

Ximg_train = np.concatenate(
    train_img_parts,
    axis=0
).astype(np.float32)

Xseq_train = np.concatenate(
    train_seq_parts,
    axis=0
).astype(np.float32)

y_train = np.concatenate(
    train_y_parts,
    axis=0
).astype(np.int32)

train_sample_weights = np.concatenate(
    train_weight_parts,
    axis=0
).astype(np.float32)

Ximg_val = np.concatenate([
    Ximg_old_val,
    Ximg_new_val,
], axis=0).astype(np.float32)

Xseq_val = np.concatenate([
    Xseq_old_val,
    Xseq_new_val,
], axis=0).astype(np.float32)

y_val = np.concatenate([
    y_old_val,
    y_new_val,
], axis=0).astype(np.int32)

val_sample_weights = np.concatenate([
    np.full(
        len(y_old_val),
        OLD_VAL_WEIGHT,
        dtype=np.float32
    ),
    np.full(
        len(y_new_val),
        NEW_VAL_WEIGHT,
        dtype=np.float32
    ),
], axis=0)

# Shuffle training arrays together.
rng_shuffle = np.random.default_rng(
    SEED + 300
)

perm = rng_shuffle.permutation(
    len(y_train)
)

Ximg_train = Ximg_train[perm]
Xseq_train = Xseq_train[perm]
y_train = y_train[perm]
train_sample_weights = (
    train_sample_weights[perm]
)

print("Final training image:", Ximg_train.shape)
print("Final training sequence:", Xseq_train.shape)
print("Final training labels:", y_train.shape)
print("Validation image:", Ximg_val.shape)
print("Validation sequence:", Xseq_val.shape)

print(
    "\nEffective original/new weighting is intentional. "
    "Test sets remain untouched and unweighted."
)

In [ ]:
# ============================================================
# STEP 21 — Backbone helpers
# ============================================================

def find_efficientnet_backbone(model):
    candidates = []

    for layer in model.layers:
        if (
            isinstance(
                layer,
                tf.keras.Model
            )
            and
            "efficientnet" in
            layer.name.lower()
        ):
            candidates.append(
                layer
            )

    if len(candidates) != 1:
        raise ValueError(
            "Expected exactly one nested EfficientNet backbone. "
            f"Found {[l.name for l in candidates]}"
        )

    return candidates[0]


def compile_refinement_model(
    model,
    learning_rate
):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=learning_rate
        ),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(),
        metrics=[
            tf.keras.metrics.SparseCategoricalAccuracy(
                name="accuracy"
            )
        ],
    )


def freeze_backbone(model):
    backbone = find_efficientnet_backbone(
        model
    )

    backbone.trainable = False

    return backbone


def unfreeze_last_n_backbone_layers(
    model,
    last_n
):
    backbone = find_efficientnet_backbone(
        model
    )

    backbone.trainable = True

    for layer in backbone.layers:
        layer.trainable = False

    tail = backbone.layers[
        -last_n:
    ]

    trainable_names = []

    for layer in tail:
        if isinstance(
            layer,
            tf.keras.layers.BatchNormalization
        ):
            layer.trainable = False
        else:
            layer.trainable = True
            trainable_names.append(
                layer.name
            )

    return (
        backbone,
        trainable_names
    )

In [ ]:
# ============================================================
# STEP 22 — Stage A
# Freeze EfficientNet backbone; adapt sequence/fusion/classifier
# ============================================================

stage_a_model = tf.keras.models.load_model(
    V1_MODEL_PATH,
    compile=False
)

stage_a_backbone = freeze_backbone(
    stage_a_model
)

compile_refinement_model(
    stage_a_model,
    STAGE_A_LR
)

print(
    "Stage A EfficientNet backbone trainable:",
    stage_a_backbone.trainable
)

stage_a_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        STAGE_A_PATH,
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1,
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        mode="min",
        restore_best_weights=True,
        verbose=1,
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1,
    ),
]

history_a = stage_a_model.fit(
    {
        "image_input": Ximg_train,
        "sequence_input": Xseq_train,
    },
    y_train,
    sample_weight=train_sample_weights,
    validation_data=(
        {
            "image_input": Ximg_val,
            "sequence_input": Xseq_val,
        },
        y_val,
        val_sample_weights,
    ),
    epochs=STAGE_A_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=stage_a_callbacks,
    verbose=1,
)

In [ ]:
# ============================================================
# STEP 23 — Stage B
# Fine-tune final EfficientNet layers at a low learning rate
# ============================================================

stage_b_model = tf.keras.models.load_model(
    STAGE_A_PATH,
    compile=False
)

(
    stage_b_backbone,
    stage_b_trainable_backbone_layers
) = unfreeze_last_n_backbone_layers(
    stage_b_model,
    FINE_TUNE_LAST_N
)

print(
    "Stage B trainable EfficientNet layers "
    "(BatchNorm excluded):"
)

for name in stage_b_trainable_backbone_layers:
    print(" -", name)

compile_refinement_model(
    stage_b_model,
    STAGE_B_LR
)

stage_b_callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        STAGE_B_PATH,
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1,
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        mode="min",
        restore_best_weights=True,
        verbose=1,
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1,
    ),
]

history_b = stage_b_model.fit(
    {
        "image_input": Ximg_train,
        "sequence_input": Xseq_train,
    },
    y_train,
    sample_weight=train_sample_weights,
    validation_data=(
        {
            "image_input": Ximg_val,
            "sequence_input": Xseq_val,
        },
        y_val,
        val_sample_weights,
    ),
    epochs=STAGE_B_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=stage_b_callbacks,
    verbose=1,
)

In [ ]:
# ============================================================
# STEP 24 — Candidate selection USING NEW VALIDATION ONLY
#
# IMPORTANT:
# The new TEST set is still untouched at this point.
# ============================================================

candidate_paths = {
    "v1.0_baseline": V1_MODEL_PATH,
    "v1.1_stageA": STAGE_A_PATH,
    "v1.1_stageB": STAGE_B_PATH,
}

candidate_val_results = {}

for name, path in candidate_paths.items():
    model = tf.keras.models.load_model(
        path,
        compile=False
    )

    result = evaluate_model(
        model,
        Ximg_new_val,
        Xseq_new_val,
        y_new_val,
        f"CANDIDATE VALIDATION — {name}"
    )

    candidate_val_results[name] = {
        "path": str(path),
        "accuracy": result["accuracy"],
        "macro_precision": result["macro_precision"],
        "macro_recall": result["macro_recall"],
        "macro_f1": result["macro_f1"],
    }

best_candidate_name = max(
    candidate_val_results,
    key=lambda name: (
        candidate_val_results[name][
            "macro_f1"
        ],
        candidate_val_results[name][
            "accuracy"
        ],
    )
)

best_candidate_path = Path(
    candidate_val_results[
        best_candidate_name
    ]["path"]
)

print("\n" + "=" * 72)
print("VALIDATION-SELECTED CANDIDATE")
print("=" * 72)
print("Selected:", best_candidate_name)
print("Path:", best_candidate_path)

PROMOTED = (
    best_candidate_name !=
    "v1.0_baseline"
)

if PROMOTED:
    shutil.copy2(
        best_candidate_path,
        FINAL_V1_1_PATH
    )

    print(
        "\nPROMOTED: a trained v1.1 candidate "
        "beat v1.0 on new validation."
    )

    print(
        "Saved:",
        FINAL_V1_1_PATH
    )

else:
    print(
        "\nNOT PROMOTED: neither fine-tuned candidate "
        "beat v1.0 on new validation."
    )

    print(
        "The stage candidates are still preserved for analysis, "
        "but v1.0 remains the recommended model."
    )

In [ ]:
# ============================================================
# STEP 25 — FINAL untouched test evaluation
# v1.0 vs validation-selected model
# ============================================================

selected_model = tf.keras.models.load_model(
    (
        FINAL_V1_1_PATH
        if PROMOTED
        else V1_MODEL_PATH
    ),
    compile=False
)

selected_model_name = (
    "v1.1_selected"
    if PROMOTED
    else "v1.0_retained"
)

selected_old_test = evaluate_model(
    selected_model,
    Ximg_old_test,
    Xseq_old_test,
    y_old_test,
    f"{selected_model_name} — ORIGINAL HELD-OUT TEST"
)

selected_new_test = evaluate_model(
    selected_model,
    Ximg_new_test,
    Xseq_new_test,
    y_new_test,
    f"{selected_model_name} — NEW REAL-HANDWRITING TEST"
)

comparison_df = pd.DataFrame([
    {
        "model": "v1.0",
        "dataset": "original_test",
        **{
            k: baseline_old_test[k]
            for k in [
                "accuracy",
                "macro_precision",
                "macro_recall",
                "macro_f1"
            ]
        }
    },
    {
        "model": "v1.0",
        "dataset": "new_real_test",
        **{
            k: baseline_new_test[k]
            for k in [
                "accuracy",
                "macro_precision",
                "macro_recall",
                "macro_f1"
            ]
        }
    },
    {
        "model": selected_model_name,
        "dataset": "original_test",
        **{
            k: selected_old_test[k]
            for k in [
                "accuracy",
                "macro_precision",
                "macro_recall",
                "macro_f1"
            ]
        }
    },
    {
        "model": selected_model_name,
        "dataset": "new_real_test",
        **{
            k: selected_new_test[k]
            for k in [
                "accuracy",
                "macro_precision",
                "macro_recall",
                "macro_f1"
            ]
        }
    },
])

display(comparison_df)

comparison_df.to_csv(
    OUTPUT_DIR /
    "v1_0_vs_v1_1_test_comparison.csv",
    index=False
)

In [ ]:
# ============================================================
# STEP 26 — Baybayin family / vowel-form confusion analysis
# ============================================================

vowels = [
    "a",
    "e_i",
    "o_u",
]

base_consonants = [
    "b", "d", "g", "h", "k",
    "l", "m", "n", "ng", "p",
    "r", "s", "t", "w", "y",
]

a_forms = [
    "ba", "da_ra", "ga", "ha", "ka",
    "la", "ma", "na", "nga", "pa",
    "ra", "sa", "ta", "wa", "ya",
]

ei_forms = [
    "be_bi", "de_di", "ge_gi", "he_hi", "ke_ki",
    "le_li", "me_mi", "ne_ni", "nge_ngi", "pe_pi",
    "re_ri", "se_si", "te_ti", "we_wi", "ye_yi",
]

ou_forms = [
    "bo_bu", "do_du", "go_gu", "ho_hu", "ko_ku",
    "lo_lu", "mo_mu", "no_nu", "ngo_ngu", "po_pu",
    "ro_ru", "so_su", "to_tu", "wo_wu", "yo_yu",
]

family_lookup = {}

for i in range(15):
    family_name = (
        f"family_{base_consonants[i]}"
    )

    for label in [
        base_consonants[i],
        a_forms[i],
        ei_forms[i],
        ou_forms[i],
    ]:
        family_lookup[label] = family_name


def family_analysis(
    y_true,
    y_pred,
    title
):
    exact_correct = 0
    consonant_total = 0
    family_correct = 0
    within_family_wrong_form = 0

    rows = []

    for true_id, pred_id in zip(
        y_true,
        y_pred
    ):
        true_label = id_to_label[
            int(true_id)
        ]

        pred_label = id_to_label[
            int(pred_id)
        ]

        exact = (
            true_label ==
            pred_label
        )

        if exact:
            exact_correct += 1

        true_family = family_lookup.get(
            true_label
        )

        pred_family = family_lookup.get(
            pred_label
        )

        same_family = (
            true_family is not None
            and
            true_family ==
            pred_family
        )

        if true_family is not None:
            consonant_total += 1

            if same_family:
                family_correct += 1

                if not exact:
                    within_family_wrong_form += 1

        rows.append({
            "true": true_label,
            "pred": pred_label,
            "exact": exact,
            "same_family": same_family,
        })

    exact_accuracy = (
        exact_correct /
        len(y_true)
    )

    family_accuracy = (
        family_correct /
        consonant_total
        if consonant_total
        else float("nan")
    )

    within_family_error_rate = (
        within_family_wrong_form /
        consonant_total
        if consonant_total
        else float("nan")
    )

    print("\n" + "=" * 72)
    print(title)
    print("=" * 72)

    print(
        f"Exact 63-class accuracy: "
        f"{exact_accuracy*100:.2f}%"
    )

    print(
        f"Consonant-family accuracy: "
        f"{family_accuracy*100:.2f}%"
    )

    print(
        "Same-family wrong-form errors: "
        f"{within_family_wrong_form}/"
        f"{consonant_total} "
        f"({within_family_error_rate*100:.2f}%)"
    )

    errors = pd.DataFrame(
        rows
    )

    same_family_errors = (
        errors[
            (~errors["exact"]) &
            (errors["same_family"])
        ]
        .groupby(
            ["true", "pred"]
        )
        .size()
        .sort_values(
            ascending=False
        )
        .rename("count")
        .reset_index()
    )

    print(
        "\nMost common same-family wrong-form confusions:"
    )

    display(
        same_family_errors.head(
            30
        )
    )

    return {
        "exact_accuracy": float(
            exact_accuracy
        ),
        "family_accuracy": float(
            family_accuracy
        ),
        "same_family_wrong_form_count": int(
            within_family_wrong_form
        ),
        "same_family_wrong_form_rate": float(
            within_family_error_rate
        ),
        "same_family_errors": same_family_errors,
    }


family_v1 = family_analysis(
    y_new_test,
    baseline_new_test["y_pred"],
    "v1.0 — NEW REAL TEST FAMILY ANALYSIS"
)

family_selected = family_analysis(
    y_new_test,
    selected_new_test["y_pred"],
    f"{selected_model_name} — NEW REAL TEST FAMILY ANALYSIS"
)

family_v1[
    "same_family_errors"
].to_csv(
    OUTPUT_DIR /
    "v1_0_same_family_errors.csv",
    index=False
)

family_selected[
    "same_family_errors"
].to_csv(
    OUTPUT_DIR /
    f"{selected_model_name}_same_family_errors.csv",
    index=False
)

In [ ]:
# ============================================================
# STEP 27 — Confusion matrices
# ============================================================

def save_confusion_matrix(
    y_true,
    y_pred,
    title,
    filename
):
    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(
            range(NUM_CLASSES)
        )
    )

    fig = plt.figure(
        figsize=(14, 14)
    )

    ax = fig.add_subplot(
        111
    )

    im = ax.imshow(
        cm
    )

    ax.set_title(
        title
    )

    ax.set_xlabel(
        "Predicted class index"
    )

    ax.set_ylabel(
        "True class index"
    )

    fig.colorbar(
        im,
        ax=ax
    )

    fig.tight_layout()

    path = (
        OUTPUT_DIR /
        filename
    )

    fig.savefig(
        path,
        dpi=180,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

    return cm, path


cm_v1_new, cm_v1_new_path = (
    save_confusion_matrix(
        y_new_test,
        baseline_new_test[
            "y_pred"
        ],
        "BAYA v1.0 — New Real-Handwriting Test Confusion Matrix",
        "v1_0_new_real_confusion_matrix.png"
    )
)

cm_selected_new, cm_selected_new_path = (
    save_confusion_matrix(
        y_new_test,
        selected_new_test[
            "y_pred"
        ],
        f"BAYA {selected_model_name} — New Real-Handwriting Test Confusion Matrix",
        f"{selected_model_name}_new_real_confusion_matrix.png"
    )
)

In [ ]:
# ============================================================
# STEP 28 — Detailed per-class reports
# ============================================================

def save_classification_report(
    y_true,
    y_pred,
    name
):
    report = classification_report(
        y_true,
        y_pred,
        labels=list(
            range(NUM_CLASSES)
        ),
        target_names=[
            id_to_label[i]
            for i in range(
                NUM_CLASSES
            )
        ],
        output_dict=True,
        zero_division=0,
    )

    path = (
        OUTPUT_DIR /
        f"{name}_classification_report.csv"
    )

    pd.DataFrame(
        report
    ).T.to_csv(
        path
    )

    return report


report_v1_new = save_classification_report(
    y_new_test,
    baseline_new_test[
        "y_pred"
    ],
    "v1_0_new_real"
)

report_selected_new = save_classification_report(
    y_new_test,
    selected_new_test[
        "y_pred"
    ],
    f"{selected_model_name}_new_real"
)

print("Saved detailed classification reports.")

In [ ]:
# ============================================================
# STEP 29 — Save histories / labels / preprocessing / metrics
# ============================================================

def history_to_jsonable(history):
    return {
        key: [
            float(v)
            for v in values
        ]
        for key, values
        in history.history.items()
    }


with open(
    OUTPUT_DIR /
    "stageA_history.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        history_to_jsonable(
            history_a
        ),
        f,
        indent=2
    )


with open(
    OUTPUT_DIR /
    "stageB_history.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        history_to_jsonable(
            history_b
        ),
        f,
        indent=2
    )


with open(
    OUTPUT_DIR /
    "Baybayin_EfficientNetB0_BiLSTM_labels_1.1.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        {
            "label_to_id":
                label_to_id,

            "id_to_label": {
                str(k): v
                for k, v
                in id_to_label.items()
            }
        },
        f,
        ensure_ascii=False,
        indent=2
    )


preprocess_config = {
    "image_height": IMG_HEIGHT,
    "image_width": IMG_WIDTH,
    "image_channels": IMG_CHANNELS,
    "image_padding_ratio": IMAGE_PADDING_RATIO,
    "ink_threshold": INK_THRESHOLD,
    "image_background_rule":
        "white background -> 0; black ink -> high values",
    "image_position_scale_normalization":
        "ink bbox -> 15% padding -> centered square -> Pillow bilinear 96x96",
    "max_sequence_length": MAX_SEQ_LEN,
    "sequence_features": [
        "x_shape",
        "y_shape",
        "dx",
        "dy",
        "dt_norm",
        "stroke_start",
    ],
    "trajectory_padding_ratio":
        TRAJECTORY_PADDING_RATIO,
    "trajectory_position_scale_normalization":
        "trajectory bbox -> centered square -> max(width,height) -> 15% padding",
    "sequence_overflow_rule":
        "stroke-preserving proportional downsampling",
    "augmentation": {
        "training_only": True,
        "synchronized_image_sequence": True,
        "rotation_degrees":
            AUG_ROTATION_DEG,
        "scale_range": [
            AUG_SCALE_MIN,
            AUG_SCALE_MAX
        ],
        "translation_pixels":
            AUG_TRANSLATION_PX,
        "horizontal_flip": False,
        "vertical_flip": False,
        "independent_kudlit_movement": False,
    },
}


with open(
    OUTPUT_DIR /
    "Baybayin_EfficientNetB0_BiLSTM_preprocessing_1.1.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        preprocess_config,
        f,
        indent=2
    )


def strip_arrays(metrics):
    return {
        key: value
        for key, value
        in metrics.items()
        if key not in {
            "probs",
            "y_pred"
        }
    }


summary = {
    "experiment": (
        "selected-model refinement; "
        "not a restart of the original model comparison"
    ),

    "seed": SEED,

    "original_model":
        V1_MODEL_PATH.name,

    "new_split_strategy":
        NEW_SPLIT_STRATEGY,

    "sample_counts": {
        "old_total":
            len(old_df),

        "new_total":
            len(new_df),

        "old_train":
            len(old_train_df),

        "old_val":
            len(old_val_df),

        "old_test":
            len(old_test_df),

        "new_train":
            len(new_train_df),

        "new_val":
            len(new_val_df),

        "new_test":
            len(new_test_df),

        "training_after_augmentation":
            len(y_train),
    },

    "augmentation": {
        "old_copies":
            AUGMENT_COPIES_PER_OLD_TRAIN_SAMPLE,

        "new_copies":
            AUGMENT_COPIES_PER_NEW_TRAIN_SAMPLE,

        "rotation_deg":
            AUG_ROTATION_DEG,

        "scale_min":
            AUG_SCALE_MIN,

        "scale_max":
            AUG_SCALE_MAX,

        "translation_px":
            AUG_TRANSLATION_PX,
    },

    "fine_tuning": {
        "stageA_epochs":
            STAGE_A_EPOCHS,

        "stageA_lr":
            STAGE_A_LR,

        "stageB_epochs":
            STAGE_B_EPOCHS,

        "stageB_lr":
            STAGE_B_LR,

        "fine_tune_last_n":
            FINE_TUNE_LAST_N,
    },

    "candidate_validation":
        candidate_val_results,

    "selected_candidate":
        best_candidate_name,

    "promoted_to_v1_1":
        PROMOTED,

    "tests": {
        "v1_0_original_test":
            strip_arrays(
                baseline_old_test
            ),

        "v1_0_new_real_test":
            strip_arrays(
                baseline_new_test
            ),

        f"{selected_model_name}_original_test":
            strip_arrays(
                selected_old_test
            ),

        f"{selected_model_name}_new_real_test":
            strip_arrays(
                selected_new_test
            ),
    },

    "family_analysis_new_test": {
        "v1_0": {
            key: value
            for key, value
            in family_v1.items()
            if key !=
            "same_family_errors"
        },

        selected_model_name: {
            key: value
            for key, value
            in family_selected.items()
            if key !=
            "same_family_errors"
        },
    },
}


with open(
    OUTPUT_DIR /
    "Baybayin_EfficientNetB0_BiLSTM_v1_1_metrics.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        summary,
        f,
        indent=2
    )

print("Saved v1.1 experiment artifacts.")

In [ ]:
# ============================================================
# STEP 30 — Save prediction CSVs for auditability
# ============================================================

def save_prediction_table(
    frame,
    y_true,
    probs,
    name
):
    y_pred = np.argmax(
        probs,
        axis=1
    )

    confidence = np.max(
        probs,
        axis=1
    )

    out = frame[
        [
            "dataset_tag",
            "source_export",
            "writer_id",
            "attempt_id",
            "label",
        ]
    ].copy().reset_index(
        drop=True
    )

    out["true_id"] = y_true

    out["pred_id"] = y_pred

    out["pred_label"] = [
        id_to_label[
            int(i)
        ]
        for i in y_pred
    ]

    out["confidence"] = (
        confidence
    )

    out["correct"] = (
        out["label"] ==
        out["pred_label"]
    )

    out.to_csv(
        OUTPUT_DIR /
        f"{name}_predictions.csv",
        index=False
    )

    return out


v1_new_predictions = (
    save_prediction_table(
        new_test_df,
        y_new_test,
        baseline_new_test[
            "probs"
        ],
        "v1_0_new_real_test"
    )
)

selected_new_predictions = (
    save_prediction_table(
        new_test_df,
        y_new_test,
        selected_new_test[
            "probs"
        ],
        f"{selected_model_name}_new_real_test"
    )
)

print("Saved prediction audit tables.")

In [ ]:
# ============================================================
# STEP 31 — Final concise report
# ============================================================

def pct(x):
    return f"{x*100:.2f}%"

print("\n" + "=" * 78)
print("BAYA EFFICIENTNET-B0–BiLSTM v1.1 REFINEMENT SUMMARY")
print("=" * 78)

print("\nNew-data split strategy:")
print(NEW_SPLIT_STRATEGY)

print("\nSamples:")
print("Original total:", len(old_df))
print("New real total:", len(new_df))
print("Training after augmentation:", len(y_train))
print("New validation:", len(new_val_df))
print("New untouched test:", len(new_test_df))

print("\nValidation-selected candidate:")
print(best_candidate_name)

print("\nPromoted to v1.1:")
print(PROMOTED)

print("\nv1.0 ORIGINAL test accuracy:")
print(
    pct(
        baseline_old_test[
            "accuracy"
        ]
    )
)

print("\nv1.0 NEW REAL test:")
print(
    "Accuracy:",
    pct(
        baseline_new_test[
            "accuracy"
        ]
    )
)
print(
    "Macro F1:",
    pct(
        baseline_new_test[
            "macro_f1"
        ]
    )
)

print(
    f"\n{selected_model_name} NEW REAL test:"
)
print(
    "Accuracy:",
    pct(
        selected_new_test[
            "accuracy"
        ]
    )
)
print(
    "Macro F1:",
    pct(
        selected_new_test[
            "macro_f1"
        ]
    )
)

print("\nSame-family wrong-form rate:")
print(
    "v1.0:",
    pct(
        family_v1[
            "same_family_wrong_form_rate"
        ]
    )
)
print(
    selected_model_name + ":",
    pct(
        family_selected[
            "same_family_wrong_form_rate"
        ]
    )
)

if "ka" in label_to_id and "ke_ki" in label_to_id:
    print(
        "\nInspect the saved prediction CSVs specifically for "
        "KA ↔ KE/KI and other kudlit/form confusions."
    )

print("\nOutput directory:")
print(OUTPUT_DIR)

if not PROMOTED:
    print(
        "\nIMPORTANT: v1.0 remains the recommended deployment model "
        "because neither trained candidate beat it on the NEW "
        "validation split."
    )

In [ ]:
# ============================================================
# STEP 32 — Package and download all artifacts
# ============================================================

archive_base = (
    ROOT /
    "BAYA_EfficientNetB0_BiLSTM_v1_1_results"
)

archive_path = shutil.make_archive(
    str(archive_base),
    "zip",
    root_dir=OUTPUT_DIR
)

print("Created:", archive_path)
print(
    "Archive size:",
    f"{Path(archive_path).stat().st_size / (1024*1024):.2f} MB"
)

from google.colab import files

files.download(
    archive_path
)

if PROMOTED and FINAL_V1_1_PATH.exists():
    print(
        "\nThe promoted final model is also inside the ZIP:"
    )
    print(
        FINAL_V1_1_PATH.name
    )
else:
    print(
        "\nNo v1.1 model was automatically promoted. "
        "Stage A/B candidates are preserved in the ZIP for analysis."
    )

# How to interpret the result

A successful run should give you **two separate stories**:

### 1. Original benchmark preservation
`v1.0` should reproduce approximately **98.10%** on the original held-out test set.

That confirms the notebook is using the correct original split and preprocessing.

### 2. Real-handwriting generalization
Compare `v1.0` and the validation-selected refinement on the **new untouched real-handwriting test set**.

The important outcomes are not only overall accuracy/Macro-F1, but also:

- `ka → ke_ki`,
- `da_ra → de_di`,
- other same-consonant-family vowel/kudlit confusions,
- the **same-family wrong-form error rate**.

## Thesis reporting rule

Do **not** replace the original 98.10% figure.

Report it as the original v1.0 result on the original held-out test set, and report the v1.1 real-handwriting refinement as a separate post-selection experiment.

If the notebook had to use the **per-class stratified fallback** because writer IDs were unavailable, do **not** call the new test an “unseen-writer” evaluation.